In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/00.Configurations

In [0]:
from pyspark.sql.functions import col,lit,when

In [0]:
silver_results_table = f'{catalog}.{silver_schema}.results'
silver_sprints_table = f'{catalog}.{silver_schema}.sprints'
gold_table = f'{catalog}.{gold_schema}.fact_session_results'

In [0]:
fact_session_results_df =  (
                              spark.table(silver_results_table).withColumn("session_type",lit("race"))
                                   .unionByName(spark.table(silver_sprints_table).withColumn("session_type",lit("sprint")))
                                   .drop(col("race_name"),col("race_date"),col("ingestion_time"),col("filename"))
                                   .withColumn("is_win",when(col("finish_position")== 1,True).otherwise(False))
                                   .withColumn("is_podium",when(col("finish_position") <= 3,True).otherwise(False))
                                   .withColumn("has_points",when(col("points")> 0,True).otherwise(False))
)

In [0]:
fact_session_results_df.write.mode("overwrite").saveAsTable(gold_table)